# Configuration

In [2]:
import os 
import pickle 

if True ^ os.getcwd().endswith('hte-and-targeting"'):
    os.chdir('..')

import logging
logger = logging.getLogger("pymc")
logger.propagate = False
logger.setLevel(logging.ERROR)

In [3]:
import random
import pandas as pd 
import numpy as np

from tqdm import tqdm

In [4]:
# visualization
import matplotlib.pyplot as plt
import seaborn as sns

# visualization params
# label size
tick_label_size = 12
legend_label_size = 12
axis_label_size = 14
title_size = 18
# font
plt.rcParams['font.family'] = 'serif'

# model label map 
model_label_map = {
    'plugin': 'Plugin', 
    'standard_bootstrap': 'Standard Bootstrap', 
    'mn_bootstrap': 'm-out-of-n Bootstrap', 
    'num_bootstrap': 'Numerical Bootstrap'
}

In [12]:
from sklearn.linear_model import LogisticRegression, Lasso

In [10]:
from core.variables import * 
import core.rct as rct
import core.targeting as tgt

# A/B Testing

## Fixed vs. Random Param DGP

### Fixed param DGP

In [5]:
optimization_params = {
    'rank_and_select': {
        'optimizer': rct.select_higher_effect,
        'params': {}, 
        'display_name': 'Rank and Select'
    }, 
}

dgp_params = {
    'n_experiments': 1, 
    'base_effects': [PointMass(1), PointMass(1.1)],
    'noise_vars': [UnivariateGaussian(0, std=1), UnivariateGaussian(0, std=1)],
    'dgp_seed': 0
}

data_params = {'sample_size': 100}

experiment_params = {'n_repeats': 500, 'fixed_params': True}

estimators_dict = {
    'bayes_normal': {
        'estimator': rct.bayes_estimate,
        'params': {'prior': 'normal', 'prior_mean': 0, 'prior_std': 1},
        'display_name': 'Shrinkage with Bayes'
    }, 
}

In [6]:
result_records = rct.repeated_experiment(
    optimization_params=optimization_params, 
    dgp_params=dgp_params, 
    data_params=data_params, 
    experiment_params=experiment_params, 
    estimators_dict=estimators_dict,
    verbose=True, 
    n_jobs=-1
)

wc_measures_dict = rct.calculate_winners_curse_measures(
    result_records=result_records, 
    optimization_params=optimization_params, 
    estimators_dict=estimators_dict, 
    data_params=data_params
)

Running 500 experiments...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    2.6s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    2.8s finished


In [7]:
delta_tau = dgp_params['base_effects'][1].mean - dgp_params['base_effects'][0].mean

for optimizer in optimization_params.keys():
    display_name = optimization_params[optimizer]['display_name']

    wc_avg = wc_measures_dict['rank_and_select_nc_wc_avg']
    wc_se = wc_measures_dict['rank_and_select_nc_wc_se'] 

    val_true_avg = wc_measures_dict['rank_and_select_nc_val_true_avg']
    val_est_avg = wc_measures_dict['rank_and_select_nc_val_est_avg']
    val_true_se = wc_measures_dict['rank_and_select_nc_val_true_se']
    val_est_se = wc_measures_dict['rank_and_select_nc_val_est_se']

    # bayesian results
    bayes_wc_avg = wc_measures_dict['rank_and_select_bayes_normal_wc_avg']
    bayes_wc_se = wc_measures_dict['rank_and_select_bayes_normal_wc_se']
    bayes_val_est_avg = wc_measures_dict['rank_and_select_bayes_normal_val_est_avg']
    bayes_val_est_se = wc_measures_dict['rank_and_select_bayes_normal_val_est_se']

    print(f'\nSelection with {display_name}:')
    print(f'  - Average True Value: {val_true_avg:.4f} ({val_true_se:.4f})')
    print(f'  - Average Estimated Value: {val_est_avg:.4f} ({val_est_se:.4f})')
    print(f'  - Average Bayes Estimated Value: {bayes_val_est_avg:.4f} ({bayes_val_est_se:.4f})')
    print()
    print(f'  - Average Winner\'s Curse (%): {wc_avg/delta_tau:.2%} ({wc_se/delta_tau:.2%})')
    print(f'  - Average Bayes Winner\'s Curse (%): {bayes_wc_avg/delta_tau:.2%} ({bayes_wc_se/delta_tau:.2%})')


Selection with Rank and Select:
  - Average True Value: 1.0764 (0.0042)
  - Average Estimated Value: 1.1196 (0.0088)
  - Average Bayes Estimated Value: 1.1086 (0.0087)

  - Average Winner's Curse (%): 43.15% (9.10%)
  - Average Bayes Winner's Curse (%): 32.22% (9.02%)


### Random param DGP

In [63]:
optimization_params = {
    'rank_and_select': {
        'optimizer': rct.select_higher_effect,
        'params': {}, 
        'display_name': 'Rank and Select'
    }, 
}

dgp_params = {
    'n_experiments': 1, 
    'base_effects': [UnivariateGaussian(1, 0.1), UnivariateGaussian(1.1, 0.1)],
    'noise_vars': [UnivariateGaussian(0, std=1), UnivariateGaussian(0, std=1)],
    'dgp_seed': 0
}

data_params = {'sample_size': 100}

experiment_params = {'n_repeats': 1000, 'fixed_params': False}

estimators_dict = {
    'bayes_normal': {
        'estimator': rct.bayes_estimate,
        'params': {'prior': 'normal', 'prior_mean': 1, 'prior_std': 1},
        'display_name': 'Shrinkage with Bayes'
    }, 
}

In [64]:
result_records = rct.repeated_experiment(
    optimization_params=optimization_params, 
    dgp_params=dgp_params, 
    data_params=data_params, 
    experiment_params=experiment_params, 
    estimators_dict=estimators_dict,
    verbose=True, 
    n_jobs=-1
)

wc_measures_dict = rct.calculate_winners_curse_measures(
    result_records=result_records, 
    optimization_params=optimization_params, 
    estimators_dict=estimators_dict, 
    data_params=data_params
)

Running 1000 experiments...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:    0.0s
[Parallel(n_jobs=-1)]: Done 1000 out of 1000 | elapsed:    0.1s finished


In [65]:
delta_tau = dgp_params['base_effects'][1].mean - dgp_params['base_effects'][0].mean

for optimizer in optimization_params.keys():
    display_name = optimization_params[optimizer]['display_name']

    wc_avg = wc_measures_dict['rank_and_select_nc_wc_avg']
    wc_se = wc_measures_dict['rank_and_select_nc_wc_se'] 

    val_true_avg = wc_measures_dict['rank_and_select_nc_val_true_avg']
    val_est_avg = wc_measures_dict['rank_and_select_nc_val_est_avg']
    val_true_se = wc_measures_dict['rank_and_select_nc_val_true_se']
    val_est_se = wc_measures_dict['rank_and_select_nc_val_est_se']

    # bayesian results
    bayes_wc_avg = wc_measures_dict['rank_and_select_bayes_normal_wc_avg']
    bayes_wc_se = wc_measures_dict['rank_and_select_bayes_normal_wc_se']
    bayes_val_est_avg = wc_measures_dict['rank_and_select_bayes_normal_val_est_avg']
    bayes_val_est_se = wc_measures_dict['rank_and_select_bayes_normal_val_est_se']

    print(f'\nSelection with {display_name}:')
    print(f'  - Average True Value: {val_true_avg:.4f} ({val_true_se:.4f})')
    print(f'  - Average Estimated Value: {val_est_avg:.4f} ({val_est_se:.4f})')
    print(f'  - Average Bayes Estimated Value: {bayes_val_est_avg:.4f} ({bayes_val_est_se:.4f})')
    print()
    print(f'  - Average Winner\'s Curse (%): {wc_avg/delta_tau:.2%} ({wc_se/delta_tau:.2%})')
    print(f'  - Average Bayes Winner\'s Curse (%): {bayes_wc_avg/delta_tau:.2%} ({bayes_wc_se/delta_tau:.2%})')


Selection with Rank and Select:
  - Average True Value: 1.1048 (0.0094)
  - Average Estimated Value: 1.1436 (0.0121)
  - Average Bayes Estimated Value: 1.1422 (0.0120)

  - Average Winner's Curse (%): 38.81% (9.25%)
  - Average Bayes Winner's Curse (%): 37.40% (9.18%)


## Negative Effects

In [6]:
optimization_params = {
    'rank_and_select': {
        'optimizer': rct.select_higher_effect,
        'params': {}, 
        'display_name': 'Rank and Select'
    }, 
}

dgp_params = {
    'n_experiments': 1, 
    'base_effects': [PointMass(-1.1), PointMass(-1)],
    'noise_vars': [UnivariateGaussian(0, std=1), UnivariateGaussian(0, std=1)],
    'dgp_seed': 0
}

data_params = {'sample_size': 100}

experiment_params = {'n_repeats': 500, 'fixed_params': True}

estimators_dict = {
    'bayes_normal': {
        'estimator': rct.bayes_estimate,
        'params': {'prior': 'normal', 'prior_mean': 0, 'prior_std': 1},
        'display_name': 'Shrinkage with Bayes'
    }, 
}

In [7]:
result_records = rct.repeated_experiment(
    optimization_params=optimization_params, 
    dgp_params=dgp_params, 
    data_params=data_params, 
    experiment_params=experiment_params, 
    estimators_dict=estimators_dict,
    verbose=True, 
    n_jobs=-1
)

wc_measures_dict = rct.calculate_winners_curse_measures(
    result_records=result_records, 
    optimization_params=optimization_params, 
    estimators_dict=estimators_dict, 
    data_params=data_params
)

Running 500 experiments...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 10 concurrent workers.
[Parallel(n_jobs=-1)]: Done  30 tasks      | elapsed:    2.7s
[Parallel(n_jobs=-1)]: Done 500 out of 500 | elapsed:    2.8s finished


In [ ]:
delta_tau = np.abs(dgp_params['base_effects'][1].mean - dgp_params['base_effects'][0].mean)

for optimizer in optimization_params.keys():
    display_name = optimization_params[optimizer]['display_name']

    wc_avg = wc_measures_dict['rank_and_select_nc_wc_avg']
    wc_se = wc_measures_dict['rank_and_select_nc_wc_se'] 

    val_true_avg = wc_measures_dict['rank_and_select_nc_val_true_avg']
    val_est_avg = wc_measures_dict['rank_and_select_nc_val_est_avg']
    val_true_se = wc_measures_dict['rank_and_select_nc_val_true_se']
    val_est_se = wc_measures_dict['rank_and_select_nc_val_est_se']

    # bayesian results
    bayes_wc_avg = wc_measures_dict['rank_and_select_bayes_normal_wc_avg']
    bayes_wc_se = wc_measures_dict['rank_and_select_bayes_normal_wc_se']
    bayes_val_est_avg = wc_measures_dict['rank_and_select_bayes_normal_val_est_avg']
    bayes_val_est_se = wc_measures_dict['rank_and_select_bayes_normal_val_est_se']

    print(f'\nPolicy Value:')
    print(f'  - True: {val_true_avg:.4f} ({val_true_se:.4f})')
    print(f'  - No Correction: {val_est_avg:.4f} ({val_est_se:.4f})')
    print(f'  - Bayesian: {bayes_val_est_avg:.4f} ({bayes_val_est_se:.4f})')
    print(f'\nWinner\'s Curse (%):')
    print(f'  - No Correction: {wc_avg/delta_tau:.2%} ({wc_se/delta_tau:.2%})')
    print(f'  - Bayesian: {bayes_wc_avg/delta_tau:.2%} ({bayes_wc_se/delta_tau:.2%})')



Selection with Rank and Select:
  - Average True Value: -1.0236 (0.0042)
  - Average Estimated Value: -0.9804 (0.0088)
  - Average Bayes Estimated Value: -0.9709 (0.0087)

  - Average Winner's Curse (%): 43.15% (9.10%)
  - Average Bayes Winner's Curse (%): 52.73% (9.02%)


# Targeting

## Negative Effects

In [25]:
optimization_params = {
    'rank_and_select': {
        'optimizer': tgt.optimize, 'params': {}, 'display_name': 'Rank and Select'
    }
}

dgp_params = {
    'base_effect_vars': [PointMass(-1.1), PointMass(-1)], 
    'cust_feat_var': UnivariateGaussian(0, 1), 
    'char_func': lambda x: x**2 + x, 
    'noise_var': UnivariateGaussian(0, 1), 
    'response_type': 'continuous', 
    'dgp_seed': 0
}

data_params = {'sample_size': 100, 'targ_sample_size': 100} 

experiment_params = {
    'n_repeats': 500, 
    'estimator': {
        'estimator': tgt.CausalForestDML, 
        'params': {
            'n_treatments': len(dgp_params['base_effect_vars']), 
            'model_t': LogisticRegression(C=0.01), 
            'model_y': Lasso(alpha=0.01), 
            'n_estimators': 100, 
            'max_depth': 5, 
            'n_jobs': -1, 
            'discrete_treatment': True
        }
    }
}

estimators_dict = {
    'standard_bootstrap': {
        'estimator': tgt.bootstrap_correction_estimate, 
        'params': {'n_bootstraps': 100, 'n_jobs': -1}, 
        'display_name': 'Standard Bootstrap'
    },
    'mn_bootstrap': {
        'estimator': tgt.bootstrap_correction_estimate, 
        'params': {'bootstrap_method': 'm_out_of_n', 'n_bootstraps': 100, 'power': 0.95, 'n_jobs': -1}, 
        'display_name': 'm-out-of-n Bootstrap'
    },
    'num_bootstrap': {
        'estimator': tgt.bootstrap_correction_estimate, 
        'params': {'bootstrap_method': 'numerical', 'n_bootstraps': 100, 'power': -0.45, 'n_jobs': -1}, 
        'display_name': 'Numerical Bootstrap'
    }, 
    'bayes_normal': {
        'estimator': tgt.bayes_estimate, 
        'params': {'prior': 'normal', 'prior_mean': 0.0, 'prior_std': 1.0},
        'display_name': 'Shrinkage'
    }, 
    'eb_spike_slab': {
        'estimator': tgt.empirical_bayes_estimate, 
        'params': {'prior': 'spike_slab', 'pi': 0.5},
        'display_name': 'EB (Spike-and-Slab)'
    }, 
}

In [ ]:
tgt_result_records = tgt.repeated_experiment(
    optimization_params=optimization_params, 
    dgp_params=dgp_params, 
    data_params=data_params, 
    experiment_params=experiment_params, 
    estimators_dict=estimators_dict, 
    verbose=True, 
    n_jobs=-1
)

tgt_wc_measures_dict = tgt.calculate_winners_curse_measures(
    result_records=tgt_result_records, 
    optimization_params=optimization_params, 
    estimators_dict=estimators_dict, 
    data_params=data_params
)

In [ ]:
delta_tau = np.abs(dgp_params['base_effect_vars'][1].mean - dgp_params['base_effect_vars'][0].mean)

for optimizer in optimization_params.keys():
    display_name = optimization_params[optimizer]['display_name']

    wc_avg = tgt_wc_measures_dict['rank_and_select_nc_wc_avg']
    wc_se = tgt_wc_measures_dict['rank_and_select_nc_wc_se'] 

    val_true_avg = tgt_wc_measures_dict['rank_and_select_nc_val_true_avg']
    val_est_avg = tgt_wc_measures_dict['rank_and_select_nc_val_est_avg']
    val_true_se = tgt_wc_measures_dict['rank_and_select_nc_val_true_se']
    val_est_se = tgt_wc_measures_dict['rank_and_select_nc_val_est_se']

    # bayesian results
    bayes_wc_avg = tgt_wc_measures_dict['rank_and_select_bayes_normal_wc_avg']
    bayes_wc_se = tgt_wc_measures_dict['rank_and_select_bayes_normal_wc_se']
    bayes_val_est_avg = tgt_wc_measures_dict['rank_and_select_bayes_normal_val_est_avg']
    bayes_val_est_se = tgt_wc_measures_dict['rank_and_select_bayes_normal_val_est_se']

    # empirical bayes results
    eb_wc_avg = tgt_wc_measures_dict['rank_and_select_eb_spike_slab_wc_avg']
    eb_wc_se = tgt_wc_measures_dict['rank_and_select_eb_spike_slab_wc_se']
    eb_val_est_avg = tgt_wc_measures_dict['rank_and_select_eb_spike_slab_val_est_avg']
    eb_val_est_se = tgt_wc_measures_dict['rank_and_select_eb_spike_slab_val_est_se']

    print(f'\nPolicy Value:')
    print(f'  - True: {val_true_avg:.4f} ({val_true_se:.4f})')
    print(f'  - No Correction: {val_est_avg:.4f} ({val_est_se:.4f})')
    print(f'  - Bayesian: {bayes_val_est_avg:.4f} ({bayes_val_est_se:.4f})')
    print(f'  - Empirical Bayes: {eb_val_est_avg:.4f} ({eb_val_est_se:.4f})')
    print(f'\nWinner\'s Curse (%):')
    print(f'  - No Correction: {wc_avg/delta_tau:.2%} ({wc_se/delta_tau:.2%})')
    print(f'  - Bayesian: {bayes_wc_avg/delta_tau:.2%} ({bayes_wc_se/delta_tau:.2%})')
    print(f'  - Empirical Bayes: {eb_wc_avg/delta_tau:.2%} ({eb_wc_se/delta_tau:.2%})')


Policy Value:
  - True: -1.0328 (0.0178)
  - No Correction: -0.8596 (0.0224)
  - Bayesian: -0.6508 (0.0185)
  - Empirical Bayes: -0.7961 (0.0207)

Winner's Curse (%):
  - No Correction: 173.25% (18.36%)
  - Bayesian: 382.03% (17.40%)
  - Empirical Bayes: 236.72% (16.95%)
